#### using the interconnections and splitting tasks

In [1]:
# runningmax abstraction with PAC guarantee example

# needed libraries
import numpy as np
import scipy.special as sp
import time
from itertools import product
import random
import gurobipy as gp
from gurobipy import GRB
from joblib import Parallel, delayed
from scipy.optimize import fsolve
from math import comb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
from scipy.optimize import brentq
from scipy.stats import truncnorm

In [2]:
N_subsys = 80 # number of subsystem considered
# for the subsystem
# choice of N for SOP
N_j = 1000 # j=1,...,N_subsys
n_dim = 1 # dimension of subsystem state set 
N_pos = 500 # N used for computing post in subsystem abstraction

# system dynamics
eta_x1, eta_w1 = 0.5, 0.5
eta_x = np.array([eta_x1]) # discretization vector
eta_ww = np.array([eta_w1 for i in range(N_subsys)])
# set of states
a1, a2 = 0, 32
a = np.array([a1])  # lower bounds
b = np.array([a2])  # upper bounds

dim1_hat = np.arange(a1+eta_x1, a2-eta_x1, eta_x1)
X_ha = np.array(list(product(dim1_hat))) 
X_haa = np.around(X_ha, 2)

dim1_hat_w = np.arange(a1+eta_w1, a2-eta_w1, eta_w1)

# Sample N_j i.i.d. pairs (x_i, x_hat_i)
x_samples = np.random.uniform(low=a, high=b, size=(N_j,1))
x_hat_indices = np.random.choice(len(X_haa), size=N_j, replace=True)
x_hat_samples = X_haa[x_hat_indices]

# Combine into a list of tuples or array of pairs
X_pairs = list(zip(x_samples, x_hat_samples))
print("susbsystem samples: ",len(X_pairs))

susbsystem samples:  1000


In [3]:
# system dynamics as black-box simulator
def sys_dyn(x,u,w):
    f_xt1 = min(0.75*(x[0] + u), w + 1, 32)
    nxt = [max(min(f_xt1, a2), a1)]
    nxt = [round(x, 2) for x in nxt]
    return nxt
        
def M_dyn(x): # intercon dynamics
    return max(x)

def generate_grid_centers(a, b, N_pos):
    dim = len(a)
    # Estimate number of points per dimension (evenly)
    k = int(np.round(N_pos ** (1 / dim)))
    total_points = k ** dim

    # Generate grid centers per dimension
    grid_axes = []
    for ai, bi in zip(a, b):
        step = (bi - ai) / k
        centers = ai + (np.arange(k) + 0.5) * step
        grid_axes.append(centers)

    # Cartesian product of all centers (i.e., subgrid centers)
    all_points = np.array(list(product(*grid_axes)))

    # Trim if more than needed
    if total_points > N_pos:
        all_points = all_points[:N_pos]

    return all_points

def Q_w(w, eta_w=eta_w1, w_min=0.0, w_max=32.0): # quantizer of internal inputs W
    w_hat = eta_w * np.round(w / eta_w)
    return np.clip(w_hat, w_min, w_max)

# for abstract subsystem
def sys_dyn_hat(y, u, w):
    y = np.asarray(y)
    X_hat_arra = np.array(X_haa)  
    a, b = y-eta_x/2, y+eta_x/2 
    # sampling N points as subgrid centres for each cell to obtain an estimate of the reachable sets as done in the paper
    sampled_points = generate_grid_centers(a, b, N_pos)
    
    # Compute successors
    successors = np.array([sys_dyn(x, u, w) for x in sampled_points])

    # Compute mean and max distance
    m = np.mean(successors, axis=0)
    r = np.max(np.abs(successors - m), axis=0)

    # Find points in X_hat_array within the under-approximation of reachable sets
    mask = np.all(np.abs(X_hat_arra - m) <= (r + eta_x / 2), axis=1)

    return X_hat_arra[mask]

# set of inputs
U, W = range(7+1), range(32+1) # subsystems external and internal input sets 
M = len(U)
U_array = np.array(U)
print("input number: ",M)

ubpr = 100
lbp, ubp = -100, 100
lbpc, ubpc = -100, 100
lbe, ube = -100, 0 

input number:  8


In [5]:
## compositional SOP
m1 = gp.Model("PAC_ASF_compositional")

vartheta = m1.addVar(vtype = GRB.CONTINUOUS, name = "vartheta", lb = lbe, ub = ube)

eps_vars = 1/len(X_pairs) 
m1.update()

gamma_b = 1e-4 # 1e-4 
delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
tau_b = 0.99 #3.001 # this is \rho in the paper
print("delta: ", delta_b)
print("epsilon: ", (delta_b/gamma_b)**0.5)

num_variables = 6 # 3  # number of coefficients as a result of the chosen degree of asf
variable_names = [f"lambda_{i}" for i in range(1, num_variables + 1)]
variable_objs = {name_i: m1.addVar(vtype=GRB.CONTINUOUS, name=name_i, lb=lbpc, ub=ubpc) for name_i in variable_names}

def Asf(x, x_hat):
    # Coefficients from the declared variables
    # c1,c2,c3 = variable_objs.values()
    c1,c2,c3,c4,c5,c6 = variable_objs.values()
    x, y = x[0], x_hat[0]
    # Polynomial expression using the variables
    result = (
        # c1 + c2*x + c3*y
        c1 + c2*x + c3*y + c4*x*y + c5*x**2 +c6*y**2 
    )
    return result

# Define lambda functions for efficiency
max_abs_diff = lambda x, y: gamma_b * np.max(np.abs(x - y))**2
asf_value = lambda x, y: Asf(x, y)

# First ASF condition subsystem level
t_init = time.time()
sum1 = gp.quicksum(eps_vars * (max_abs_diff(x, y) - asf_value(x, y)) for (x,y) in X_pairs)
m1.addConstr(sum1 <= vartheta)
t_fin = time.time()
diff_t = t_fin - t_init
print("done1: ",diff_t)
m1.update()

t_init = time.time()

X0 = [X_pairs[l][0] for l in range(len(X_pairs))]
X1 = [X_pairs[l][1] for l in range(len(X_pairs))]
U_list = list(U)
W_list = list(W)

nL = len(X_pairs)
nU = len(U_list)
nW = len(W_list)

def compute_dynamics(x0, x1, u, w, r):
    dyn_hat_out = sys_dyn_hat(x1, u, Q_w(w))  # quantized noise
    dyn_out     = sys_dyn(x0, u, w)            # true noise
    return dyn_out, list(dyn_hat_out)

dyn_results = Parallel(n_jobs=-1, backend="threading", verbose=2)(
    delayed(compute_dynamics)(X0[l], X1[l], U_list[i], W_list[r], r)
    for l in range(nL)
    for i in range(nU)
    for r in range(nW)
)

# 3D index: result[l, i, r] -> dyn_results[l*nU*nW + i*nW + r]
m1.addConstrs(
    (
        gp.quicksum(
            asf_value(dyn_results[l*nU*nW + i*nW + r][0], yn) / len(dyn_results[l*nU*nW + i*nW + r][1])
            for yn in dyn_results[l*nU*nW + i*nW + r][1]
        )
        - tau_b * asf_value(X0[l], X1[l]) + (tau_b - 1) * delta_b <= vartheta
        for l in range(nL)
        for i in range(nU)
        for r in range(nW)
    ),
    name="asf2"
)

m1.update()
t_fin = time.time()
print(f"Setup time: {t_fin - t_init:.2f}s")

# def compute_sys_dyn(l, i, r):
#     return l, i, r, sys_dyn_hat(X_pairs[l][1], U[i], Q_w(W[r])), sys_dyn(X_pairs[l][0], U[i], W[r])

# # Step 1: Compute system dynamics in parallel
# num_jobs = -1  # Use all available cores
# dyn_results = Parallel(n_jobs=num_jobs, verbose=2)(
#     delayed(compute_sys_dyn)(l, i, r) for l in range(len(X_pairs)) for i in range(len(U)) for r in range(len(W))
# )

# # Step 2: Add constraints sequentially (Gurobi constraint addition must be sequential during parallelization)
# for l, i, r, dyn_hat_output, dyn_output in dyn_results:
#     m1.addConstr(
#         gp.quicksum(1/(len(dyn_hat_output)) * asf_value(dyn_output, yn) for yn in dyn_hat_output) # sigma taken uniformly
#         - tau_b*asf_value(X_pairs[l][0], X_pairs[l][1]) + (tau_b-1)*delta_b <= vartheta
#     )

# t_fin = time.time()
# diff_t = t_fin - t_init
m1.update()
print("done2: ",diff_t)

m1.Params.LogToConsole = 0
m1.setParam("NumericFocus", 3)
m1.setParam('NonConvex', 2)

m1.update()
print("Start now")
print("Number of variables:", m1.numVars)
print("Number of constraints:", m1.numConstrs)
t_init = time.time()

m1.setObjective(vartheta, GRB.MINIMIZE)
m1.setParam('Presolve', 0)

m1.optimize()
t_fin = time.time()
diff_t = t_fin - t_init
print("opt_status:", m1.status)

m1.write("comp_with_PAC_Infeasible.lp")

# Check if optimization was successful
if m1.status == GRB.OPTIMAL:
    with open('comp_with_PAC_variables.txt', 'w') as file:
        for v in m1.getVars():
            file.write(f'{v.VarName} {v.X}\n')
elif m1.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m1.computeIIS()
    m1.write("comp_with_PAC_Infeasible.ilp")
    print("IIS written to subsys_with_PAC_Infeasible.ilp")

    with open("comp_with_PAC_Infeasible.txt", "w") as f:
        for c in m1.getConstrs():
            if c.IISConstr:
                f.write(f"Infeasible constraint: {c.ConstrName}\n")
        for v in m1.getVars():
            if v.IISLB or v.IISUB:
                f.write(f"Infeasible bound on variable: {v.VarName}\n")

else:
    print("Optimization was not successful (not optimal or infeasible). Status:", m1.status)

# Count binding constraints
sup_const = [constr for constr in m1.getConstrs() if abs(constr.Slack) < 1e-4]
s_ = len(sup_const)
print(f"Number of constraints within tol: {s_}")
print("Time (in s):", diff_t)

delta:  2.5e-05
epsilon:  0.5
done1:  0.033560991287231445


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 345 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 628 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 993 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done 1438 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done 1965 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 2572 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 3261 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done 4881 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 5812 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 6825 tasks      | elapsed:    9.0s
[Parallel(n_jobs=-1)]: Done 7918 tasks      | elapsed:   10.4s
[Parallel(n_jobs=-1)]: Done 9093 tasks   

Setup time: 367.56s
done2:  0.033560991287231445
Start now
Number of variables: 7
Number of constraints: 264001
opt_status: 2
Number of constraints within tol: 95
Time (in s): 0.2971329689025879


In [6]:
def prune_constraints_inf_norm(m1, tol=1e-4, n_jobs=-1):
    # Solve original model once
    m1.optimize()
    if m1.status != gp.GRB.OPTIMAL:
        raise RuntimeError("Original model is not optimal.")

    sol_star = np.array([v.X for v in m1.getVars()])

    # --- Free pre-filter: skip constraints with large slack (clearly non-binding) ---
    C = list(m1.getConstrs())
    candidates = []
    for constr in C:
        slack = constr.Slack
        # Only test constraints that are near-active (slack ≈ 0)
        if abs(slack) <= tol * 10:
            candidates.append(constr)
    
    print(f"Total constraints: {len(C)}, candidates to test: {len(candidates)}")

    # --- Parallel testing of candidate constraints ---
    # We pass the model as a file to avoid pickling Gurobi objects
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".mps", delete=False) as f:
        tmp_path = f.name
    m1.write(tmp_path)

    # Get constraint names for candidates
    candidate_names = [c.ConstrName for c in candidates]
    sol_star_copy = sol_star.copy()

    def test_constraint(constr_name):
        import gurobipy as gp_local
        import numpy as np_local

        env = gp_local.Env()
        env.setParam("OutputFlag", 0)  # suppress output
        env.setParam("LogToConsole", 0)

        m_copy = gp_local.read(tmp_path, env)
        m_copy.setParam("OutputFlag", 0)

        c = m_copy.getConstrByName(constr_name)
        if c is None:
            return constr_name, False
        m_copy.remove(c)
        m_copy.update()
        m_copy.optimize()

        if m_copy.status == gp_local.GRB.OPTIMAL:
            sol_new = np_local.array([v.X for v in m_copy.getVars()])
            diff = np_local.linalg.norm(sol_star_copy - sol_new, ord=np_local.inf)
            return constr_name, diff > tol
        return constr_name, False  # infeasible/unbounded → not necessary per your logic

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=2)(
        delayed(test_constraint)(name) for name in candidate_names
    )

    os.unlink(tmp_path)  # clean up temp file

    # Collect necessary constraints
    necessary_names = {name for name, is_necessary in results if is_necessary}
    necessary_constraints = [c for c in C if c.ConstrName in necessary_names]

    return necessary_constraints

# Usage
s_N = len(prune_constraints_inf_norm(m1, tol=1e-4))
print(f"Number of support constraints: {s_N}")

Total constraints: 264001, candidates to test: 225


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.


Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - f

[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    7.5s
/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic lic

[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:   35.4s


Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter Username
Set parameter LicenseID to value 2778218
Set para

[Parallel(n_jobs=-1)]: Done 225 out of 225 | elapsed:   56.4s finished


#### nonconvex SOP PAC bound
Consider Equation (7) in https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8299432

#### comparison of $\beta,\alpha$, and $\mathcal{N}$.

In [7]:
# as alternative, consider eqn 7 in the paper
def epsil(k,N1,beta1):
    res = (beta1/(N1*comb(N1, k)))**(1/(N1-k))
    return 1-res

In [20]:
beta_subsys = 10**-6
# alpha_sN_subsys = epsil(s_N_comp, N_j, beta_subsys)
alpha_sN_subsys = epsil(1, 5000, beta_subsys)

print(f"With confidence of {(1-beta_subsys)*100:.6f}%, the non-violation of at least {1-alpha_sN_subsys:.6f}, and violation {alpha_sN_subsys:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.993848, and violation 0.006152


In [12]:
N_subsys, N_0, N_j = 4, 1000, 1000
N_bar = N_0 + N_subsys*N_j
alpha_bar = (N_subsys + 1)*alpha_sN_subsys
beta_bar = (N_subsys + 1)*beta_subsys
print("Compositional result")
print(f"Compositional samples: {(N_bar):.6f}")
print(f"With confidence of {(1-beta_bar)*100:.6f}%, the non-violation of at least {1-alpha_bar:.6f}, and violation {alpha_bar:.6f}")

Compositional result
Compositional samples: 5000.000000
With confidence of 99.999500%, the non-violation of at least 0.863602, and violation 0.136398


In [9]:
def epsil_nonconvex(k, N1, beta1):
    """
    Compute PAC violation probability epsilon for the non-convex scenario bound.
    
    Solves:
        sum_{i=0}^k C(N1,i) * eps^i * (1-eps)^(N1-i) = beta1
    """

    def f(eps):
        s = 0.0
        for i in range(k+1):
            s += comb(N1, i) * (eps**i) * ((1-eps)**(N1-i))
        return s - beta1

    # epsilon must lie in (0,1)
    eps = brentq(f, 1e-12, 1-1e-12)
    
    return eps


In [27]:
beta_subsys = 10**-6
N_j = 9000
# alpha_sN_subsys = epsil(s_N_comp, N_j, beta_subsys)
alpha_sN_subsys = epsil_nonconvex(1, N_j, beta_subsys)

print(f"With confidence of {(1-beta_subsys)*100:.6f}%, the non-violation of at least {1-alpha_sN_subsys:.6f}, and violation {alpha_sN_subsys:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.998147, and violation 0.001853


In [28]:
N_subsys = 80
N_bar = N_subsys*N_j
alpha_bar = (N_subsys)*alpha_sN_subsys
beta_bar = (N_subsys)*beta_subsys
print("Compositional result")
print(f"Compositional samples: {(N_bar):.6f}")
print(f"With confidence of {(1-beta_bar)*100:.6f}%, the non-violation of at least {1-alpha_bar:.6f}, and violation {alpha_bar:.6f}")

Compositional result
Compositional samples: 720000.000000
With confidence of 99.992000%, the non-violation of at least 0.851788, and violation 0.148212
